# 1.3 PAC learning

Probably Approximately Correct: with enough samples, ERM is within epsilon of the best, with probability at least one minus delta.

PAC turns the ERM worry into a sample-size guarantee with accuracy and confidence knobs. This is a gap topic in the lesson plan, so the notebook anchors the missing application content to the finite-class formula and then contrasts it with the agnostic scaling.

Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(13)


def pac_m(H_size, epsilon, delta):
    value = (math.log(H_size) + math.log(1 / delta)) / epsilon
    return int(math.ceil(value))


def agnostic_m(H_size, epsilon, delta):
    value = (math.log(H_size) + math.log(1 / delta)) / (2 * epsilon ** 2)
    return int(math.ceil(value))


def finite_class_rung(name, H_sizes, epsilon, delta, mode="realizable"):
    rows = []
    for H_size in H_sizes:
        if mode == "realizable":
            m = pac_m(H_size, epsilon, delta)
        else:
            m = agnostic_m(H_size, epsilon, delta)
        rows.append({
            "name": name,
            "H_size": int(H_size),
            "epsilon": float(epsilon),
            "delta": float(delta),
            "m": int(m),
            "mode": mode,
        })
    return rows


def menu_points(H_size):
    columns = int(np.ceil(np.sqrt(H_size)))
    rows = int(np.ceil(H_size / columns))
    coords = []
    for idx in range(H_size):
        coords.append((idx % columns, idx // columns))
    return np.asarray(coords, dtype=float)

## The concept, built once (D1)

For a finite realizable class, PAC sample complexity is

$$m\gerac{\ln|H|+\ln(1/\delta)}{arepsilon}.$$

The lesson's main calculation uses $|H|=100$, $arepsilon=0.1$, and $\delta=0.05$.

In [ ]:
lesson_m = pac_m(100, 0.1, 0.05)
print("required samples:", lesson_m)

The exact plan number is $77$. The same function should also give $93$ for $\delta=0.01$, $153$ when $arepsilon=0.05$, and $123$ when $|H|=10{,}000$.

In [ ]:
assert lesson_m == 77
assert pac_m(100, 0.1, 0.01) == 93
assert pac_m(100, 0.05, 0.05) == 153
assert pac_m(10000, 0.1, 0.05) == 123
extra_per_doubling = math.log(2) / 0.1
assert np.isclose(round(extra_per_doubling, 2), 6.93)
print("extra samples per class doubling:", round(extra_per_doubling, 2))

## The dataset ladder

This is a capacity and knob ladder: D1 is the lesson finite class, D2 sweeps $|H|$, D3 tightens $arepsilon$, D4 tightens $\delta$, and D5 uses the agnostic $1/arepsilon^2$ contrast because the realizable assumption fails.

In [ ]:
rungs = []
rungs.append(finite_class_rung("D1 lesson finite class", [100], 0.1, 0.05))
rungs.append(finite_class_rung("D2 growing class", [50, 100, 200, 400], 0.1, 0.05))
rungs.append([
    {"name": "D3 tighten epsilon", "H_size": 100, "epsilon": eps, "delta": 0.05, "m": pac_m(100, eps, 0.05), "mode": "realizable"}
    for eps in [0.2, 0.1, 0.05, 0.025]
])
rungs.append([
    {"name": "D4 tighten delta", "H_size": 100, "epsilon": 0.1, "delta": delta, "m": pac_m(100, 0.1, delta), "mode": "realizable"}
    for delta in [0.2, 0.1, 0.05, 0.01]
])
rungs.append([
    {"name": "D5 agnostic contrast", "H_size": 100, "epsilon": eps, "delta": 0.05, "m": agnostic_m(100, eps, 0.05), "mode": "agnostic"}
    for eps in [0.2, 0.1, 0.05, 0.025]
])

for rows in rungs:
    print(rows[0]["name"])
    print("  rows=", len(rows))
    print("  preview=", rows[:3])

In [ ]:
summary = []
for rows in rungs:
    max_m = max(row["m"] for row in rows)
    summary.append((rows[0]["name"], max_m))
print("rung | max required sample size")
for name, max_m in summary:
    print(f"{name}: {max_m}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for ax, rows in zip(axes[0], rungs):
    H_size = rows[-1]["H_size"]
    coords = menu_points(min(H_size, 400))
    ax.scatter(coords[:, 0], coords[:, 1], s=8)
    ax.set_title(f"{rows[0]['name']}
menu size shown={min(H_size, 400)}")
    ax.set_xticks([])
    ax.set_yticks([])
for ax, rows in zip(axes[1], rungs):
    if rows[0]["name"].startswith("D2"):
        x = [row["H_size"] for row in rows]
        ax.set_xscale("log", base=2)
        ax.set_xlabel("|H|")
    elif rows[0]["name"].startswith("D3") or rows[0]["name"].startswith("D5"):
        x = [row["epsilon"] for row in rows]
        ax.invert_xaxis()
        ax.set_xlabel("epsilon")
    elif rows[0]["name"].startswith("D4"):
        x = [row["delta"] for row in rows]
        ax.invert_xaxis()
        ax.set_xlabel("delta")
    else:
        x = [row["H_size"] for row in rows]
        ax.set_xlabel("|H|")
    y = [row["m"] for row in rows]
    ax.plot(x, y, marker="o")
    ax.set_ylabel("required m")
    ax.set_title("Payoff curve")
    ax.grid(True, alpha=0.3)
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: using the realizable formula in the agnostic case. When no hypothesis is perfect, the lesson warns that the dependence becomes $1/arepsilon^2$, not $1/arepsilon$.

In [ ]:
epsilon = 0.05
wrong = pac_m(100, epsilon, 0.05)
fixed = agnostic_m(100, epsilon, 0.05)
print(f"wrong realizable count={wrong}")
print(f"fixed agnostic scaling count={fixed}")
print(f"multiplicative increase={fixed / wrong:.1f}x")

## Evaluate it + Practice

- Metric: required sample size m.
- No-skill baseline: reuse the D1 sample count for every epsilon and delta.
- Cheap sanity check: each class doubling at epsilon 0.1 adds about 6.93 samples before rounding.
- Ablation: replace agnostic_m with pac_m on D5 and understate the budget.
- Failure signals: infinite |H|, non-realizable labels, or a requested epsilon below the noise floor.

### Practice prompts

- Compute the sample count for |H|=1000, epsilon=0.1, delta=0.05.

- Find the extra samples needed when delta changes from 0.05 to 0.005.

- Compare realizable and agnostic counts at epsilon=0.02.